In [ ]:
%load_ext watermark


In [ ]:
import os
import subprocess

os.environ["POLARS_FORCE_NEW_STREAMING"] = "1"

from IPython.display import display, HTML
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm

from pylib._seed_global_rngs import seed_global_rngs


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = "2025-05-19-freqscreen-summary"
teeplot_subdir


In [ ]:
seed_global_rngs(1)


## Get Data


In [ ]:
data_sources = {
    # "dummy": "https://osf.io/cjqx2/download",
    "uk-big": "https://osf.io/pgj7t/download",
    "uk": "https://osf.io/mkjy5/download",
    "multistrain": "https://osf.io/ywmpt/download",
    "vanilla": "https://osf.io/r8skg/download",
    "vanilla-big": "https://osf.io/j4795/download",
    "vanilla-big-1.3x": "https://osf.io/cnp5z/download",
}
tmp_path = f"/tmp/{teeplot_subdir}.pqt"


In [ ]:
results = []

for source_name, url in data_sources.items():
    print(f"Downloading {source_name} data from {url}")

    subprocess.run(
        [
            "wget",
            "--tries=5",
            "--show-progress",
            "--progress=bar:force",
            "-O",
            str(tmp_path),
            url,
        ],
        check=True,
    )
    print("done!")

    df = pl.scan_parquet(
        tmp_path,
        low_memory=True,
        retries=5,
    )

    unique_groups = (
        df.unique(
            [
                "trt_name",
                "trt_n_downsample",
                "trt_hsurf_bits",
                "replicate_uuid",
            ]
        )
        .select(
            pl.col("trt_name"),
            pl.col("trt_n_downsample"),
            pl.col("trt_hsurf_bits"),
            pl.col("replicate_uuid"),
        )
        .drop_nans()
        .drop_nulls()
        .collect(engine="streaming")
    )

    for (trt_name, trt_n_downsample, trt_hsurf_bits, replicate_uuid) in tqdm(
        [*unique_groups.iter_rows()],
    ):
        phylo_df = (
            df.filter(
                (pl.col("trt_name") == trt_name)
                & (pl.col("trt_n_downsample") == trt_n_downsample)
                & (pl.col("trt_hsurf_bits") == trt_hsurf_bits)
                & (pl.col("replicate_uuid") == replicate_uuid)
            )
            .collect(engine="streaming")
            .to_pandas()
        )

        num_muts = phylo_df["defmut_mask_sum"].astype(bool).sum()
        num_focal_muts = phylo_df["is_focal_defmut"].sum()
        num_bg_sites = phylo_df["cfg_maxseqlen"].unique().item()

        expected_num_muts = (
            phylo_df["defmut_mask_sum"].sum() - num_focal_muts
        ) / num_bg_sites

        lambda_poisson = expected_num_muts
        p_poisson = scipy_stats.poisson.sf(num_focal_muts, lambda_poisson)

        results.append(
            {
                "source_name": source_name,
                "trt_name": trt_name,
                "trt_n_downsample": trt_n_downsample,
                "trt_hsurf_bits": trt_hsurf_bits,
                "replicate_uuid": replicate_uuid,
                "num_muts": num_muts,
                "num_focal_muts": num_focal_muts,
                "num_bg_sites": num_bg_sites,
                "expected_num_muts": expected_num_muts,
                "p_poisson": p_poisson,
                "lambda_poisson": lambda_poisson,
            }
        )


In [ ]:
results_df = pd.DataFrame(results)
results_df


In [ ]:
results_df["trt_name_"] = results_df["trt_name"].str.replace("/", "\n")


In [ ]:
bins = [-np.inf, 0.005, 0.01, 0.05, 0.1, np.inf]
labels = ["<0.005", "<0.01", "<0.05", "<0.1", "ns"]

for y in ("p_poisson",):
    display(HTML(f"<h1>{y=}</h1>"))

    data = results_df[
        (results_df["trt_hsurf_bits"] == 0)
        & (
            results_df["trt_n_downsample"]
            == results_df
            .groupby(["trt_name","source_name"])["trt_n_downsample"]
            .transform("max")
        )
    ].copy()
    data["sig"] = pd.cut(
        data[y],
        bins=bins,
        labels=labels,
        include_lowest=True,
    )
    data["sig"] = data["sig"].cat.set_categories(labels, ordered=True)
    col_order = filter(
        set(data["trt_name_"]).__contains__,
        [
            "Sneu\nGneu",
            "Sben1.1x\nGneu",
            "Sben1.3x\nGneu",
            "Sben2x\nGneu",
            "Sben1.1x\nGdel1.1x",
            "Sben1.3x\nGdel1.3x",
            "Sben2x\nGdel2x",
        ],
    )
    col_order = [*col_order]
    with tp.teed(
        sns.catplot,
        data=data,
        y=y,
        col="trt_name_",
        row="source_name",
        hue="sig",
        col_order=col_order,
        legend=False,
        kind="swarm",
        dodge=True,
        margin_titles=True,
        alpha=0.5,
        height=1.6,
        aspect=0.7,
        size=4,
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.set_titles(col_template="{col_name}", row_template="{row_name}")
        for ax in teed.axes.flat:
            ax.set_yscale("symlog", linthresh=0.0001)
            ax.set_ylim(-0.0001, None)
            ax.axhline(0.5, color="black", linestyle=":")
            ax.axhline(0.05, color="red", linestyle="--")
            ax.tick_params(axis="x", rotation=-60)
            ax.set_xlabel("")
            ax.set_ylabel(y.replace("_", " "))

        teed.tight_layout()

    for source_name, group_df in data.groupby("source_name"):
        with tp.teed(
            sns.catplot,
            data=group_df,
            y=y,
            col="trt_name_",
            hue="sig",
            col_order=filter(
               set(group_df["trt_name_"]).__contains__,
               col_order,
            ),
            legend=False,
            kind="swarm",
            dodge=True,
            margin_titles=True,
            alpha=0.5,
            height=1.6,
            aspect=0.7,
            size=4,
            teeplot_outattrs={"source_name": source_name},
            teeplot_subdir=teeplot_subdir,
        ) as teed:
            teed.set_titles(col_template="{col_name}", row_template="{row_name}")
            for ax in teed.axes.flat:
                ax.set_yscale("symlog", linthresh=0.0001)
                ax.set_ylim(-0.0001, None)
                ax.axhline(0.5, color="black", linestyle=":")
                ax.axhline(0.05, color="red", linestyle="--")
                ax.tick_params(axis="x", rotation=-60)
                ax.set_xlabel("")
                ax.set_ylabel(y.replace("_", " "))

            teed.tight_layout()


In [ ]:
data = results_df[
    (results_df["trt_hsurf_bits"] == 0)
    & (
        results_df["trt_n_downsample"]
        == results_df
        .groupby(["trt_name","source_name"])["trt_n_downsample"]
        .transform("max")
    )
]

with tp.teed(
    sns.catplot,
    data=data,
    y="num_focal_muts",
    col="trt_name_",
    row="source_name",
    col_order=col_order,
    legend=False,
    kind="swarm",
    dodge=True,
    margin_titles=True,
    alpha=0.5,
    height=2,
    aspect=0.7,
    size=4,
    teeplot_subdir=teeplot_subdir,
) as teed:
    teed.set_titles(col_template="{col_name}", row_template="{row_name}")
    for ax in teed.axes.flat:
        ax.set_yscale("log")

        ax.set_xlabel("")

    teed.tight_layout()

for source_name, group_df in data.groupby("source_name"):
    with tp.teed(
        sns.catplot,
        data=group_df,
        y="num_focal_muts",
        col="trt_name_",
        col_order=filter(set(group_df["trt_name_"]).__contains__, col_order),
        legend=False,
        kind="swarm",
        dodge=True,
        margin_titles=True,
        alpha=0.5,
        height=2,
        aspect=0.7,
        size=4,
        teeplot_outattrs={"source_name": source_name},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.set_titles(col_template="{col_name}", row_template="{row_name}")
        for ax in teed.axes.flat:
            ax.set_yscale("log")

            ax.set_xlabel("")
